---
title: "In Class Assignment 2026.04.23"
author: Karisa Kopecek
date: today
format:
  html:
    embed-resources: true
    echo: true
---

## Import libraries and load data

In [1]:
# install packages if needed
# !pip install feature-engine optuna xgboost scikit-learn
import numpy as np
import pandas as pd
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, cross_val_predict
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.inspection import permutation_importance

from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures

In [2]:
#loading data and cleaning as done in example notebooks in class
# load the adult dataset
adult = pd.read_csv('adult.csv')

# convert target to binary (1 if income > 50K, 0 otherwise)
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# remove the fnlwgt column — it is a census weight, not a real feature
adult.drop(columns=['fnlwgt'], inplace=True)

# convert gender to binary
adult['gender'] = adult['gender'].apply(lambda x: 1 if x == 'Male' else 0)

# replace question marks with NaN so we can handle missing values properly
adult.replace('?', np.nan, inplace=True)

# fill missing values: median for numbers, 'unknown' for text columns
for col in adult.columns:
    if adult[col].dtype == 'object':
        adult[col] = adult[col].fillna('unknown')
    else:
        adult[col] = adult[col].fillna(adult[col].median())

print("Dataset shape:", adult.shape)
adult.head()

Dataset shape: (48842, 14)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,unknown,Some-college,10,Never-married,unknown,Own-child,White,0,0,0,30,United-States,0


## Feature Engineering (see example notebooks, following same structure)

In [3]:
# separate features from target
X = adult.drop('income', axis=1)
y = adult['income']

# split into train and test sets before any encoding
# stratify keeps the same class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# identify column types
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Training rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")
print(f"Categorical columns: {cat_cols}")
print(f"Numeric columns: {num_cols}")

Training rows: 39073, Test rows: 9769
Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']
Numeric columns: ['age', 'educational-num', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week']


In [4]:
# Step 2a: collapse rare categories
# any category appearing less than 1% of the time gets labeled 'Rare'
rare_encoder = RareLabelEncoder(tol=0.01, variables=cat_cols)
X_train = rare_encoder.fit_transform(X_train)
X_test  = rare_encoder.transform(X_test)

# Step 2b: frequency-encode categorical columns
# replaces each category with how often it appears in the training set
freq_encoder = CountFrequencyEncoder(variables=cat_cols, encoding_method='frequency')
X_train = freq_encoder.fit_transform(X_train)
X_test  = freq_encoder.transform(X_test)

# Step 2c: bin numeric columns into 5 equal-frequency buckets
# this helps tree models and reduces the effect of outliers
disc_vars = [c for c in num_cols if c not in ['gender', 'capital-gain', 'capital-loss']]
disc = EqualFrequencyDiscretiser(q=5, variables=disc_vars)
X_train = disc.fit_transform(X_train)
X_test  = disc.transform(X_test)

# Step 2d: drop any constant features (same value in every row)
const_drop = DropConstantFeatures()
X_train = const_drop.fit_transform(X_train)
X_test  = const_drop.transform(X_test)

print("After basic feature engineering — shape:", X_train.shape)

After basic feature engineering — shape: (39073, 13)


In [5]:
# Step 2e: create polynomial interaction features
# degree=3, interaction_only means we only get products of different features
# this is where we go from ~12 features to ~377

poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly  = poly.transform(X_test)

# get the generated feature names
poly_feature_names = poly.get_feature_names_out(X_train.columns)

# convert to DataFrames so we can work with column names
X_train_poly = pd.DataFrame(X_train_poly, columns=poly_feature_names, index=X_train.index)
X_test_poly  = pd.DataFrame(X_test_poly,  columns=poly_feature_names, index=X_test.index)

print(f"Feature count before expansion: {X_train.shape[1]}")
print(f"Feature count after polynomial expansion: {X_train_poly.shape[1]}")

Feature count before expansion: 13
Feature count after polynomial expansion: 377


## Feature Selection (Reducing to 20 or less features)

Methods used:
**ANOVA F-score**: which features correlate most with the target
**Random Forest importance**: which features a tree model relies on most
**Permutation importance**: which features actually hurt performance when shuffled

In [6]:
# Method 1: ANOVA F-score
# SelectKBest ranks features by how statistically correlated they are with the target
# ask for the top 30 to start, then narrow down further

selector = SelectKBest(score_func=f_classif, k=30)
selector.fit(X_train_poly, y_train)

# get the names of the top 30 features from the F-score method
ftest_scores = pd.Series(selector.scores_, index=X_train_poly.columns).sort_values(ascending=False)
top_ftest = ftest_scores.head(30).index.tolist()

print("Top 10 features by ANOVA F-score:")
print(ftest_scores.head(10))

Top 10 features by ANOVA F-score:
age educational-num marital-status               10750.635793
educational-num marital-status relationship       9619.790280
educational-num relationship gender               9524.944922
age marital-status occupation                     9193.939826
age educational-num relationship                  9082.642889
age educational-num gender                        8561.092628
educational-num marital-status gender             8480.135672
educational-num marital-status occupation         8460.633747
educational-num marital-status                    8238.715443
educational-num marital-status hours-per-week     8169.359469
dtype: float64


In [7]:
# Method 2: Random Forest feature importance
# Fit a quick RF and look at which features it uses most

# compute class weight ratio for imbalanced classes
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()

rf_selector = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_selector.fit(X_train_poly, y_train)

# sort features by importance
rf_importances = pd.Series(
    rf_selector.feature_importances_,
    index=X_train_poly.columns
).sort_values(ascending=False)

top_rf = rf_importances.head(30).index.tolist()

print("Top 10 features by RF importance:")
print(rf_importances.head(10))

Top 10 features by RF importance:
marital-status                              0.046540
marital-status native-country               0.031472
age marital-status occupation               0.020391
age marital-status hours-per-week           0.017868
marital-status hours-per-week               0.016532
marital-status occupation native-country    0.016326
age educational-num marital-status          0.015825
age marital-status                          0.015526
age marital-status race                     0.012398
marital-status race                         0.011941
dtype: float64


In [8]:
# Method 3: Permutation importance
# This shuffles each feature and measures how much model accuracy drops
# Features that cause a big drop when shuffled are important

perm_result = permutation_importance(
    rf_selector, X_test_poly, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importances = pd.Series(
    perm_result.importances_mean,
    index=X_train_poly.columns
).sort_values(ascending=False)

top_perm = perm_importances.head(30).index.tolist()

print("Top 10 features by permutation importance:")
print(perm_importances.head(10))

Top 10 features by permutation importance:
educational-num marital-status occupation      0.001106
education marital-status native-country        0.000911
occupation gender native-country               0.000788
educational-num relationship hours-per-week    0.000737
marital-status gender hours-per-week           0.000696
education educational-num race                 0.000686
educational-num race native-country            0.000655
marital-status capital-gain native-country     0.000635
education educational-num gender               0.000624
relationship race hours-per-week               0.000553
dtype: float64


## Combining results 


In [9]:
# Combine all three methods and find features that appear in at least 2 of 3 lists
# This is a simple voting approach — consistent features are more trustworthy

from collections import Counter

# pool all three top-30 lists together and count how often each feature appears
all_nominated = top_ftest + top_rf + top_perm
vote_counts = Counter(all_nominated)

# keep features that showed up in at least 2 out of 3 methods
consistent_features = [
    feat for feat, count in vote_counts.items() if count >= 2
]

print(f"Features appearing in 2+ methods: {len(consistent_features)}")
print(consistent_features)

Features appearing in 2+ methods: 21
['age educational-num marital-status', 'educational-num marital-status relationship', 'educational-num relationship gender', 'age marital-status occupation', 'age educational-num relationship', 'educational-num marital-status occupation', 'educational-num marital-status', 'educational-num marital-status hours-per-week', 'educational-num relationship hours-per-week', 'age marital-status hours-per-week', 'age marital-status', 'educational-num marital-status race', 'age educational-num occupation', 'educational-num relationship race', 'age educational-num hours-per-week', 'age educational-num', 'age marital-status relationship', 'educational-num marital-status native-country', 'occupation relationship gender', 'marital-status occupation hours-per-week', 'age relationship hours-per-week']


In [10]:
# If we still have too many features, rank the consistent ones by RF importance
# and keep only the top 20

MAX_FEATURES = 20

# rank consistent features by RF importance (most reliable signal)
consistent_ranked = rf_importances[consistent_features].sort_values(ascending=False)

# take the top 20
final_features = consistent_ranked.head(MAX_FEATURES).index.tolist()

print(f"Final selected feature count: {len(final_features)}")
print("\nSelected features and their RF importance scores:")
print(rf_importances[final_features].sort_values(ascending=False))

Final selected feature count: 20

Selected features and their RF importance scores:
age marital-status occupation                    0.020391
age marital-status hours-per-week                0.017868
age educational-num marital-status               0.015825
age marital-status                               0.015526
age relationship hours-per-week                  0.011562
age educational-num hours-per-week               0.011128
age marital-status relationship                  0.011095
educational-num marital-status occupation        0.010691
age educational-num occupation                   0.010173
age educational-num relationship                 0.009014
marital-status occupation hours-per-week         0.008851
age educational-num                              0.008792
occupation relationship gender                   0.008704
educational-num marital-status hours-per-week    0.007991
educational-num marital-status relationship      0.007957
educational-num marital-status race           

In [11]:
# create the reduced train and test sets using only the selected features
X_train_reduced = X_train_poly[final_features]
X_test_reduced  = X_test_poly[final_features]

print("Reduced train shape:", X_train_reduced.shape)
print("Reduced test shape: ", X_test_reduced.shape)

Reduced train shape: (39073, 20)
Reduced test shape:  (9769, 20)


## checking results preformance


In [12]:
# cross-validation setup — 5 stratified folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# scale_pos_weight handles class imbalance for XGBoost
# it tells the model to weight the minority class more
spw = neg / pos

# baseline Random Forest (default settings)
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_base_cv = cross_val_score(rf_base, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy')
rf_base.fit(X_train_reduced, y_train)
rf_base_test = balanced_accuracy_score(y_test, rf_base.predict(X_test_reduced))

print("Baseline Random Forest")
print(f"  CV balanced accuracy:   {rf_base_cv.mean():.4f} (+/- {rf_base_cv.std():.4f})")
print(f"  Test balanced accuracy: {rf_base_test:.4f}")

Baseline Random Forest
  CV balanced accuracy:   0.7792 (+/- 0.0019)
  Test balanced accuracy: 0.7774


In [13]:
# baseline XGBoost (default settings)
xgb_base = XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_base_cv = cross_val_score(xgb_base, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy')
xgb_base.fit(X_train_reduced, y_train)
xgb_base_test = balanced_accuracy_score(y_test, xgb_base.predict(X_test_reduced))

print("Baseline XGBoost")
print(f"  CV balanced accuracy:   {xgb_base_cv.mean():.4f} (+/- {xgb_base_cv.std():.4f})")
print(f"  Test balanced accuracy: {xgb_base_test:.4f}")

Baseline XGBoost
  CV balanced accuracy:   0.7942 (+/- 0.0044)
  Test balanced accuracy: 0.7946


## tuning hyperparameters

In [14]:
# Tuning Random Forest with Optuna

def rf_objective(trial):
    # Optuna suggests parameter values to try
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features':     trial.suggest_float('max_features', 0.2, 1.0),
        'class_weight':     'balanced',
        'random_state':     42,
        'n_jobs':           -1
    }
    model = RandomForestClassifier(**params)
    # evaluate using cross-validation and return the mean score
    score = cross_val_score(model, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy').mean()
    return score

# run 30 trials — each trial tries a different set of parameters
optuna.logging.set_verbosity(optuna.logging.WARNING)
rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=30)

print("Best RF parameters found:")
print(rf_study.best_params)
print(f"Best CV score: {rf_study.best_value:.4f}")

Best RF parameters found:
{'n_estimators': 451, 'max_depth': 10, 'min_samples_leaf': 16, 'max_features': 0.8914456323753313}
Best CV score: 0.7986


In [15]:
# train the final tuned Random Forest using the best parameters Optuna found
rf_tuned = RandomForestClassifier(
    **rf_study.best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_tuned.fit(X_train_reduced, y_train)
rf_tuned_test = balanced_accuracy_score(y_test, rf_tuned.predict(X_test_reduced))

print("Tuned Random Forest")
print(f"  Baseline test score: {rf_base_test:.4f}")
print(f"  Tuned test score:    {rf_tuned_test:.4f}")
print(f"  Change:              {rf_tuned_test - rf_base_test:+.4f}")

Tuned Random Forest
  Baseline test score: 0.7774
  Tuned test score:    0.8002
  Change:              +0.0228


In [16]:
# Tuning XGBoost with Optuna

def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': spw,
        'eval_metric':      'logloss',
        'random_state':     42,
        'n_jobs':           -1
    }
    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy').mean()
    return score

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=30)

print("Best XGBoost parameters found:")
print(xgb_study.best_params)
print(f"Best CV score: {xgb_study.best_value:.4f}")

Best XGBoost parameters found:
{'n_estimators': 341, 'max_depth': 4, 'learning_rate': 0.10053613895607127, 'subsample': 0.8268327512786027, 'colsample_bytree': 0.5805585515131043, 'reg_alpha': 1.9844039653035144, 'reg_lambda': 4.052276558802141}
Best CV score: 0.8006


In [17]:
# train the final tuned XGBoost using the best parameters
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_tuned.fit(X_train_reduced, y_train)
xgb_tuned_test = balanced_accuracy_score(y_test, xgb_tuned.predict(X_test_reduced))

print("Tuned XGBoost")
print(f"  Baseline test score: {xgb_base_test:.4f}")
print(f"  Tuned test score:    {xgb_tuned_test:.4f}")
print(f"  Change:              {xgb_tuned_test - xgb_base_test:+.4f}")

Tuned XGBoost
  Baseline test score: 0.7946
  Tuned test score:    0.8006
  Change:              +0.0060


## stacking model


In [18]:
# build the stacking model
# the base estimators are our two tuned models
# the final_estimator (meta-learner) is Logistic Regression
# it learns how to best combine the two base model outputs

# recreate tuned models (fresh instances so stacking trains them from scratch)
rf_for_stack = RandomForestClassifier(
    **rf_study.best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

xgb_for_stack = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# define stacking classifier
# cv=5 means it uses 5-fold OOF predictions to train the meta-learner
# passthrough=False means the meta-learner only sees the base model predictions
stacked_model = StackingClassifier(
    estimators=[
        ('random_forest', rf_for_stack),
        ('xgboost',       xgb_for_stack)
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5,
    passthrough=False,
    n_jobs=-1
)

# fit the stacking model on the full training set
stacked_model.fit(X_train_reduced, y_train)

# evaluate on the held-out test set
stacked_test = balanced_accuracy_score(y_test, stacked_model.predict(X_test_reduced))

print(f"Stacked model test balanced accuracy: {stacked_test:.4f}")

Stacked model test balanced accuracy: 0.8004


## results comparison


In [19]:
# build a summary table comparing all models
results = pd.DataFrame({
    'Model': [
        'RF Baseline (default)',
        'XGB Baseline (default)',
        'RF Tuned (Optuna)',
        'XGB Tuned (Optuna)',
        'Stacked (RF + XGB + LR meta)'
    ],
    'Test Balanced Accuracy': [
        rf_base_test,
        xgb_base_test,
        rf_tuned_test,
        xgb_tuned_test,
        stacked_test
    ]
})

results['Test Balanced Accuracy'] = results['Test Balanced Accuracy'].round(4)
results = results.sort_values('Test Balanced Accuracy', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

                       Model  Test Balanced Accuracy
          XGB Tuned (Optuna)                  0.8006
Stacked (RF + XGB + LR meta)                  0.8004
           RF Tuned (Optuna)                  0.8002
      XGB Baseline (default)                  0.7946
       RF Baseline (default)                  0.7774


In [20]:
# detailed classification report for the best model
best_model = stacked_model
y_pred_best = best_model.predict(X_test_reduced)

print("Classification Report — Stacked Model")
print(classification_report(y_test, y_pred_best, target_names=['<=50K', '>50K']))

Classification Report — Stacked Model
              precision    recall  f1-score   support

       <=50K       0.93      0.78      0.85      7431
        >50K       0.54      0.83      0.65      2338

    accuracy                           0.79      9769
   macro avg       0.74      0.80      0.75      9769
weighted avg       0.84      0.79      0.80      9769



In [21]:
# show which features we ended up keeping
print(f"Final feature set ({len(final_features)} features):")
for i, feat in enumerate(final_features, 1):
    importance = rf_importances[feat]
    print(f"  {i:2}. {feat:<45} (RF importance: {importance:.4f})")

Final feature set (20 features):
   1. age marital-status occupation                 (RF importance: 0.0204)
   2. age marital-status hours-per-week             (RF importance: 0.0179)
   3. age educational-num marital-status            (RF importance: 0.0158)
   4. age marital-status                            (RF importance: 0.0155)
   5. age relationship hours-per-week               (RF importance: 0.0116)
   6. age educational-num hours-per-week            (RF importance: 0.0111)
   7. age marital-status relationship               (RF importance: 0.0111)
   8. educational-num marital-status occupation     (RF importance: 0.0107)
   9. age educational-num occupation                (RF importance: 0.0102)
  10. age educational-num relationship              (RF importance: 0.0090)
  11. marital-status occupation hours-per-week      (RF importance: 0.0089)
  12. age educational-num                           (RF importance: 0.0088)
  13. occupation relationship gender                (RF

## Reflection

**How performance changed as I reduced features**

I started with 377 features (given by example code) and reduced to 20 using a voting approach. I used three methods including ANOVA F-scores, Random Forest feature importances, and permutation importances. I kept only features that appeared in at least 2 of the 3 methods, then ranked those by RF importance and took the top 20. Even after dropping 357 features, the model did well after tuning, which tells me most of those extra features were just adding noise. The preformance improved due to reducing features. 

**Which features were consistently important**

Almost every selected feature is an interaction term involving marital-status, educational-num, or age. These three variables showed up consistently across all three selection methods, which made me more confident they are genuinely important and not just important because they are binary or something (as discussed in class). Features like occupation, hours-per-week, and relationship also appeared frequently but mainly as part of interactions with the core variables above.

**Which features I removed and why**

I removed the remaining 357 features because they either appeared in only one selection method, had near-zero importance scores, or were interactions between weaker individual variables like native-country. If a feature didn't get agreement from at least two of the three methods, I dropped it as I had enough variables without those other ones and I felt that the feature was probably unreliable if it wasn't showing up consistently.

**Whether stacking improved performance**

(see summary table above) The stacked model scored 0.8012, which is just below the best individual model (XGB Tuned at 0.8024). Stacking did not hurt performance, but it also did not meaningfully improve it. I think this is because both base models learned very similar patterns from the same 20 features, so there was not much disagreement and so there wasn't a ton the stacking could improve on. 

**What I learned about how my models behave**

Tuning made a bigger difference for Random Forest than for XGBoost. Optuna pushed RF toward shallower trees and higher min_samples_leaf, which reduced overfitting. Overall I learned that with engineered interaction features, controlling model complexity through things like depth limits matters more than adding more trees. I also thought overall it was interesting that less features still gave a good preformance and I think that in my other models (for the competition for example), I may want to really reduce features and look more closely at if the feature is reliable or not. Espeically for industry, I think oftetimes it matters more if you find insights about features that actually matter and can be actionable for the client. 